# Equivalent sites on a circle
The model<sup>1,2</sup> describes jumps between $N$ equivalent sites on a circle of radius $R$, for example a methyl group ($N = 3$) rotating about its axis. The scattering is an elastic line plus $N - 1$ Lorentzians:

$$ S(Q, E) = A_0(Q)\,\delta(E) + \sum_{l=1}^{N-1} A_l(Q)\, L(E, \Gamma_l) $$

with

$$ A_l(Q) = \frac{1}{N} \sum_{k=1}^{N} j_0(Q r_k) \cos\left(\frac{2\pi l k}{N}\right),
\quad r_k = 2R \sin\left(\frac{\pi k}{N}\right),
\quad \Gamma_l = \frac{2\hbar}{\tau} \sin^2\left(\frac{\pi l}{N}\right), $$

where $\tau$ is the residence time on a site and $j_0(x) = \sin(x)/x$. $A_0$ is the Elastic Incoherent Structure Factor (EISF), $A_l$ is the Quasi-Elastic Incoherent Structure Factor (QISF) of order $l$, and they sum to one. The widths do not depend on $Q$; only the weights do.

The width is also written as $4 \sin^2(\pi l / N) / \tau$, with a $\tau$ that is twice the residence time used here.<sup>3</sup>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import easydynamics as edyn

%matplotlib widget

## The model at different Q values

In [ ]:
# Q is in Angstrom^-1, the radius in Angstrom and the residence time in ps.
Q = np.linspace(0.5, 2, 7)

circle_model = edyn.EquivalentSitesCircle(
    scale=1.0,
    radius=1.0,
    residence_time=2.0,
    number_of_sites=3,
    Q=Q,
)
circle_model

With $N = 3$ sites the model has $N - 1 = 2$ Lorentzians per $Q$. The radius is fixed by default, since it is normally known from the molecular geometry; free it with `circle_model.radius.fixed = False` to fit it.

In [ ]:
# Plot the quasi-elastic part (all Lorentzians) at every Q. The delta function
# would be a spike at E = 0; its weight, the EISF, is plotted further down.
energy = np.linspace(-2, 2, 501)
component_collections = circle_model.get_component_collections()

cmap = plt.cm.jet
nQ = len(component_collections)
plt.figure()
for Q_index, collection in enumerate(component_collections):
    quasielastic = sum(
        component.evaluate(energy)
        for component in collection
        if isinstance(component, edyn.Lorentzian)
    )
    color = cmap(Q_index / (nQ - 1))
    plt.plot(energy, quasielastic, label=f'Q={Q[Q_index]:.2f} Å^-1', color=color)

plt.legend()
plt.xlabel('Energy (meV)')
plt.ylabel('Intensity (arb. units)')
plt.title('Equivalent sites on a circle, N = 3')
plt.show()

## The widths of the Lorentzians

In [ ]:
for order in range(1, circle_model.number_of_lorentzians + 1):
    width = circle_model.calculate_width(order=order)[0]
    print(f'Lorentzian of order {order}: HWHM = {width:.4f} meV')

For $N = 3$ both orders have the same width, $(2\hbar/\tau)\sin^2(\pi/3) = (2\hbar/\tau)\sin^2(2\pi/3)$, so the quasi-elastic part is a single Lorentzian. For larger $N$ the orders have different widths.

## EISF and QISF as a function of Q

In [ ]:
Q_fine = np.linspace(0.01, 4, 200)

plt.figure()
plt.plot(Q_fine, circle_model.calculate_EISF(Q_fine), label='EISF, $A_0$')
for order in range(1, circle_model.number_of_lorentzians + 1):
    plt.plot(
        Q_fine,
        circle_model.calculate_QISF_order(Q_fine, order=order),
        label=f'QISF of order {order}, $A_{order}$',
    )
plt.plot(
    Q_fine,
    circle_model.calculate_EISF(Q_fine) + circle_model.calculate_QISF(Q_fine),
    color='k',
    linestyle='dashed',
    label='EISF + QISF',
)
plt.xlabel('Q (Å$^{-1}$)')
plt.ylabel('Weight')
plt.title('Weights for N = 3')
plt.legend()
plt.show()

The EISF depends on the number of sites. For $N = 2$ this is the two-site jump model with jump distance $2R$; for large $N$ it approaches continuous rotational diffusion on the circle.

In [ ]:
plt.figure()
for number_of_sites in [2, 3, 6]:
    model = edyn.EquivalentSitesCircle(
        radius=1.0, residence_time=2.0, number_of_sites=number_of_sites
    )
    plt.plot(Q_fine, model.calculate_EISF(Q_fine), label=f'N = {number_of_sites}')
plt.xlabel('Q (Å$^{-1}$)')
plt.ylabel('EISF')
plt.title('EISF for different numbers of sites')
plt.legend()
plt.show()

## Components and fit targets
`EquivalentSitesCircle` is built on `DeltaLorentzianDiffusionModelBase`, the same base class as `JumpSitesLogNormalDistribution` and `GaussianModel3D`. Every collection holds a delta function and the Lorentzians of order $l$, named `<lorentzian_name>_l<l>`. Their areas and widths are dependent Parameters, so they follow the model parameters during a fit.

In [ ]:
collection = circle_model.get_component_collections()[0]
print([component.name for component in collection])
print([fit_target.name for fit_target in circle_model.get_fit_targets()])

## References

1. J. D. Barnes, J. Chem. Phys. **58**, 5193 (1973), [doi:10.1063/1.1679130](https://doi.org/10.1063/1.1679130).

2. M. Bée, *Quasielastic Neutron Scattering*, Adam Hilger, Bristol (1988).

3. Mantid, [`InelasticDiffRotDiscreteCircle`](https://github.com/mantidproject/mantid/blob/6ec66a6f524a7ec9d4258351fe3001c66c63dbe5/Framework/CurveFitting/src/Functions/InelasticDiffRotDiscreteCircle.cpp#L79-L108): its `Decay` parameter is this $2\tau$.